## **Pertemuan 4 - Dashboard Statistik Deskriptif dengan Streamlit**
By: Farhan dan Samuel

### Tujuan Pembelajaran
Setelah pertemuan ini, mahasiswa diharapkan mampu:
1. Menjelaskan **cara kerja Streamlit** (skrip dijalankan ulang setiap ada interaksi).
2. Menjalankan aplikasi Streamlit dari terminal dan memakai **elemen, widget, dan layout** dasarnya.
3. Menampilkan **ukuran statistik deskriptif** (mean, median, modus, std, kuartil, IQR, outlier) secara interaktif.
4. Membangun **dashboard multi-tab** lengkap dengan filter, grafik, dan simulasi dadu.
5. Menerapkan prinsip dashboard yang baik (audiens, jumlah komponen, load time, tidak menyesatkan).

### Alur Materi (90 menit)
| Waktu | Bagian | Topik |
|---|---|---|
| 15 menit | 0 - 2 | Persiapan, cara kerja Streamlit, Hello Streamlit |
| 20 menit | 3 - 4 | Elemen, widget, layout, dan *rerun* |
| 30 menit | 5 - 6 | Statistik interaktif dan **dashboard lengkap** (live coding) |
| 25 menit | 7 - 9 | Prinsip dashboard, troubleshooting, **kuis latihan** |

### Bagaimana notebook ini dipakai?
> **PENTING:** Streamlit **tidak dijalankan di dalam notebook**. Notebook ini dipakai untuk **membaca penjelasan** dan **menulis file `.py`** lewat perintah `%%writefile`. Setelah file terbentuk, aplikasinya dijalankan dari **terminal / command prompt**.
>
> Karena itu, jalankan notebook ini di komputer sendiri (Jupyter / VS Code), **bukan Google Colab**.

## 0. Persiapan

### 0.1 Struktur folder
Buat satu folder kerja (misal `pert4/`) dan letakkan notebook ini di dalamnya, bersama folder `data/` dari pengajar:

```
pert4/
├── Pert4_Modul_Streamlit.ipynb    <- notebook ini
└── data/
    ├── data_mahasiswa.csv
    └── data_penjualan_kafe.csv
```
File `.py` yang kita buat nanti akan muncul otomatis **di folder yang sama**.

### 0.2 Install library
Buka terminal / command prompt, lalu jalankan:
```
pip install -U streamlit pandas numpy matplotlib
```
Cek hasilnya dengan cell di bawah ini:

In [ ]:
import importlib.util
import os

for lib in ["streamlit", "pandas", "numpy", "matplotlib"]:
    status = "OK" if importlib.util.find_spec(lib) else "BELUM TERINSTALL"
    print(f"{lib:12s}: {status}")

print()
print("Folder data/ :", "OK" if os.path.exists("data/data_mahasiswa.csv") else "TIDAK DITEMUKAN")

## 1. Apa Itu Streamlit dan Bagaimana Cara Kerjanya?

**Streamlit** adalah framework Python untuk membuat **aplikasi web interaktif** hanya dengan Python, tanpa perlu HTML, CSS, atau JavaScript. Cocok untuk dashboard data karena kompatibel dengan NumPy, Pandas, dan Matplotlib yang sudah kita pelajari.

### Konsep paling penting: *rerun*

```
 Pengguna menggeser slider / menekan tombol
              │
              ▼
 Streamlit menjalankan ULANG seluruh skrip .py dari baris pertama sampai terakhir
              │
              ▼
 Tampilan di browser diperbarui
```

Artinya:
- Skrip Streamlit **bukan** program yang berjalan sekali lalu selesai, melainkan dijalankan ulang **setiap kali** ada interaksi.
- Variabel biasa **akan kembali ke nilai awal** setiap rerun. Untuk menyimpan nilai antar-rerun dipakai `st.session_state`.
- Untuk menghindari pekerjaan berat berulang (misalnya membaca file besar), dipakai `@st.cache_data`.

Kita akan membuktikan kedua hal ini sendiri di Bagian 4 dan 5.

## 2. Aplikasi Pertama: Hello Streamlit

Jalankan cell di bawah. Cell ini **tidak menjalankan aplikasi**, hanya **menulis file** `01_hello.py`.

In [ ]:
%%writefile 01_hello.py
import streamlit as st

st.title("Hello, Streamlit! 👋")
st.write("Ini aplikasi web pertamaku, dibuat hanya dengan Python.")

nama = st.text_input("Siapa namamu?")

if nama:
    st.success(f"Halo, {nama}! Selamat datang di StatPro 1.")

Sekarang buka **terminal** di folder yang sama dan jalankan:
```
streamlit run 01_hello.py
```
Browser akan terbuka otomatis di alamat `http://localhost:8501`. Coba ketik namamu di kotak input.

**Cara menghentikan aplikasi:** tekan `Ctrl + C` di terminal.

> Kalau muncul pesan *"streamlit is not recognized / command not found"*, pakai: `python -m streamlit run 01_hello.py`

#### Eksperimen kecil
1. Biarkan aplikasi tetap berjalan, lalu **ubah teks** di `01_hello.py` (misal judulnya), simpan filenya.
2. Kembali ke browser, klik **Rerun** (atau pilih *Always rerun*) di pojok kanan atas. Tampilan langsung berubah, tanpa perlu menjalankan ulang dari terminal.

## 3. Elemen Dasar: Teks, Data, dan Grafik

Elemen adalah komponen untuk **menampilkan** output (bersifat pasif). Ringkasannya:

| Kelompok | Fungsi yang sering dipakai |
|---|---|
| Teks | `st.title()`, `st.header()`, `st.subheader()`, `st.markdown()`, `st.caption()`, `st.latex()`, `st.code()` |
| Data | `st.dataframe()` (interaktif), `st.table()` (statis), `st.metric()` |
| Grafik | `st.pyplot(fig)` untuk Matplotlib, `st.line_chart()`, `st.bar_chart()` |
| Pesan | `st.success()`, `st.info()`, `st.warning()`, `st.error()` |

In [ ]:
%%writefile 02_elemen.py
import matplotlib.pyplot as plt
import pandas as pd
import streamlit as st

st.title("Tur Elemen Streamlit")

# ---------- 1. Teks ----------
st.header("1. Elemen Teks")
st.subheader("Ini subheader")
st.markdown("Teks **tebal**, *miring*, dan `kode`.")
st.caption("Caption: teks kecil untuk keterangan.")
st.latex(r"\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i")
st.code("rata_rata = sum(data) / len(data)", language="python")

# ---------- 2. Tampilan data ----------
st.header("2. Tampilan Data")
df = pd.DataFrame({
    "Nama": ["Andi", "Budi", "Citra", "Dewi"],
    "UTS": [78, 65, 88, 92],
})

st.dataframe(df)            # tabel interaktif (bisa di-sort)
st.table(df.head(2))        # tabel statis
st.metric(label="Rata-rata UTS", value=f"{df['UTS'].mean():.1f}", delta="+2.5")

# ---------- 3. Grafik ----------
st.header("3. Grafik Matplotlib")
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(df["Nama"], df["UTS"])
ax.set_ylabel("Nilai UTS")
ax.set_title("Nilai UTS per Mahasiswa")
st.pyplot(fig)

Jalankan dengan `streamlit run 02_elemen.py` dan perhatikan:
- `st.dataframe` bisa diurutkan dengan klik judul kolom, sedangkan `st.table` tidak.
- `st.metric` cocok untuk menampilkan **angka kunci** (nilai statistik) pada dashboard.
- `st.pyplot(fig)` menampilkan grafik Matplotlib yang sama seperti di Pertemuan 2.

## 4. Widget, Layout, dan Membuktikan *Rerun*

**Widget** adalah komponen yang menerima **input** dari pengguna. Hasilnya langsung berupa nilai Python biasa.

| Widget | Nilai yang dikembalikan |
|---|---|
| `st.text_input("Nama")` | string |
| `st.number_input("Umur")` | angka |
| `st.slider("Bin", 5, 30, 10)` | angka |
| `st.selectbox("Pilih", [...])` | satu elemen dari list |
| `st.multiselect("Pilih", [...])` | list elemen yang dipilih |
| `st.checkbox("Aktifkan")` | `True` / `False` |
| `st.button("Klik")` | `True` **hanya pada saat diklik** |
| `st.file_uploader("Upload")` | file yang diunggah (atau `None`) |

**Layout** mengatur tata letak halaman:

| Layout | Fungsi |
|---|---|
| `st.sidebar` | panel samping, tempat ideal untuk filter dan pengaturan |
| `st.columns(n)` | membagi halaman menjadi n kolom berdampingan |
| `st.tabs([...])` | beberapa tampilan dalam satu halaman |
| `st.expander("...")` | bagian yang bisa dibuka-tutup |

In [ ]:
%%writefile 03_widget_layout.py
import streamlit as st

st.title("Widget dan Layout")

# ---------- Sidebar: tempat widget input ----------
st.sidebar.header("Pengaturan")
nama = st.sidebar.text_input("Nama", "Mahasiswa")
semester = st.sidebar.selectbox("Semester", [1, 2, 3, 4, 5, 6, 7, 8], index=2)
minat = st.sidebar.multiselect(
    "Minat",
    ["Statistika", "Machine Learning", "Visualisasi", "Basis Data"],
    default=["Statistika"],
)
jumlah = st.sidebar.slider("Jumlah data", 10, 100, 30)
detail = st.sidebar.checkbox("Tampilkan detail", value=True)

st.write(f"Halo **{nama}** (semester {semester})")

# ---------- Columns: tata letak berdampingan ----------
kol1, kol2, kol3 = st.columns(3)
kol1.metric("Semester", semester)
kol2.metric("Jumlah minat", len(minat))
kol3.metric("Jumlah data", jumlah)

# ---------- Tabs: pindah tampilan tanpa pindah halaman ----------
tab1, tab2 = st.tabs(["Detail", "Tombol"])

with tab1:
    if detail:
        st.write("Minat kamu:", minat)
    else:
        st.info("Centang 'Tampilkan detail' di sidebar.")

with tab2:
    if st.button("Klik aku!"):
        st.balloons()
        st.write("Tombol ditekan 🎉")

# ---------- Eksperimen: Streamlit menjalankan ulang seluruh skrip ----------
st.divider()
st.subheader("Eksperimen: skrip dijalankan ulang setiap interaksi")

hitung = 0
if st.button("Tambah (variabel biasa)"):
    hitung += 1
st.write("Variabel biasa:", hitung)

if "klik" not in st.session_state:
    st.session_state["klik"] = 0
if st.button("Tambah (session_state)"):
    st.session_state["klik"] += 1
st.write("session_state:", st.session_state["klik"])

Jalankan `streamlit run 03_widget_layout.py`, lalu **lakukan eksperimen di bagian paling bawah**:

1. Klik tombol **"Tambah (variabel biasa)"** beberapa kali. Berapa nilai yang tampil?
2. Klik tombol **"Tambah (session_state)"** beberapa kali. Berapa nilai yang tampil?

**Pertanyaan diskusi:**
- Mengapa variabel biasa selalu kembali ke 0 atau 1, sedangkan `session_state` terus bertambah?
- *Jawaban:* setiap klik menyebabkan skrip dijalankan ulang dari awal, sehingga `hitung = 0` dieksekusi lagi. `st.session_state` menyimpan nilai **di luar** siklus rerun.

## 5. Statistik Deskriptif Interaktif

Saatnya menggabungkan Pertemuan 2 (Pandas dan statistika deskriptif) dengan Streamlit. Perhatikan dua hal baru di kode berikut:

1. **`@st.cache_data`**: fungsi pembaca data hanya dijalankan **sekali**; rerun berikutnya memakai hasil yang tersimpan. Ini menjaga *load time* tetap singkat.
2. **`.dropna()`**: kolom `Kehadiran` punya data kosong, dan perhitungan harus tetap aman.

In [ ]:
%%writefile 04_statistik.py
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import streamlit as st

DATA_PATH = Path(__file__).parent / "data" / "data_mahasiswa.csv"


@st.cache_data          # data dibaca sekali saja, bukan setiap interaksi
def muat_data(path):
    return pd.read_csv(path)


df = muat_data(DATA_PATH)

st.title("Statistik Deskriptif Interaktif")

kolom = st.selectbox("Pilih kolom numerik", ["UTS", "UAS", "Jam_Belajar", "Kehadiran"])
x = df[kolom].dropna()          # buang data kosong

c1, c2, c3, c4 = st.columns(4)
c1.metric("Mean", f"{x.mean():.2f}")
c2.metric("Median", f"{x.median():.2f}")
c3.metric("Std Dev (sampel)", f"{x.std():.2f}")
c4.metric("Jumlah data (n)", len(x))

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(x, bins=10, edgecolor="black")
ax.axvline(x.mean(), color="red", linestyle="--", label="Mean")
ax.axvline(x.median(), color="green", label="Median")
ax.set_xlabel(kolom)
ax.set_ylabel("Frekuensi")
ax.legend()
st.pyplot(fig)

Jalankan `streamlit run 04_statistik.py`, lalu ganti-ganti kolom di dropdown.

**Pertanyaan diskusi:**
- Kolom mana yang mean-nya paling berbeda dari median? Apa artinya terhadap bentuk histogramnya?
- Mengapa jumlah data (n) pada `Kehadiran` lebih kecil dari kolom lain?

## 6. Dashboard Lengkap (Live Coding)

Sekarang kita bangun dashboard utuh. Struktur kodenya terdiri dari **4 blok**:

| Blok | Isi | Komponen kunci |
|---|---|---|
| 1. Konfigurasi | judul tab browser, layout lebar | `st.set_page_config` |
| 2. Fungsi bantu | `hitung_statistik()` mengembalikan dictionary berisi semua ukuran | function dan dict (Pertemuan 1) |
| 3. Sidebar | upload CSV, pilih kolom, filter kategori | `file_uploader`, `selectbox`, `multiselect` |
| 4. Halaman utama | 4 tab: Data, Statistik, Visualisasi, Simulasi Dadu | `tabs`, `columns`, `metric`, `pyplot` |

### Wireframe dashboard
```
┌──────────────┬─────────────────────────────────────────────┐
│  SIDEBAR     │  📊 Dashboard Statistik Deskriptif           │
│              │  [📋 Data][🧮 Statistik][📈 Visual][🎲 Dadu] │
│ • Upload CSV │ ┌────────┬────────┬────────┬────────┐        │
│ • Kolom      │ │  Mean  │ Median │ Modus  │   n    │        │
│ • Filter     │ ├────────┼────────┼────────┼────────┤        │
│   Kelas      │ │  Std   │ Range  │  IQR   │Outlier │        │
│   Jurusan    │ └────────┴────────┴────────┴────────┘        │
│              │  ✅ Mean ≈ Median → distribusi simetris       │
└──────────────┴─────────────────────────────────────────────┘
```

### Tips live coding
Jangan salin seluruh kode sekaligus. Ketik **bertahap**, dan jalankan setelah setiap tahap:
1. Konfigurasi + fungsi bantu + muat data, lalu tampilkan `st.dataframe(df)`.
2. Tambahkan **sidebar** (pilih kolom dan filter).
3. Tambahkan **tab Statistik** (`st.metric`).
4. Tambahkan **tab Visualisasi** (histogram dan boxplot).
5. Tambahkan **tab Simulasi Dadu**.

Kode lengkap sebagai referensi akhir (jalankan cell untuk membuat `app.py`):

In [ ]:
%%writefile app.py
"""
Dashboard Statistik Deskriptif - Statistik dan Probabilitas 1 (Pertemuan 4)

Jalankan dari terminal (di dalam folder ini):
    streamlit run app.py
"""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

# ======================================================================
# 1. KONFIGURASI HALAMAN
# ======================================================================
st.set_page_config(page_title="Dashboard StatPro 1", page_icon="📊", layout="wide")

DATA_DEFAULT = Path(__file__).parent / "data" / "data_mahasiswa.csv"


# ======================================================================
# 2. FUNGSI BANTU
# ======================================================================
@st.cache_data
def muat_data_default(path):
    """Baca CSV bawaan. Di-cache supaya tidak dibaca ulang setiap interaksi."""
    return pd.read_csv(path)


def hitung_statistik(x: pd.Series) -> dict:
    """Hitung seluruh ukuran statistik deskriptif untuk satu kolom."""
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr
    return {
        "n": len(x),
        "mean": x.mean(),
        "median": x.median(),
        "modus": list(x.mode()),
        "std": x.std(),                       # sampel (ddof=1)
        "min": x.min(),
        "max": x.max(),
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "batas_bawah": batas_bawah,
        "batas_atas": batas_atas,
        "outlier": x[(x < batas_bawah) | (x > batas_atas)],
    }


def format_modus(daftar, maks_tampil=3):
    teks = ", ".join(f"{m:g}" for m in daftar[:maks_tampil])
    return teks + (" ..." if len(daftar) > maks_tampil else "")


# ======================================================================
# 3. SIDEBAR: SUMBER DATA, PILIH KOLOM, FILTER
# ======================================================================
st.sidebar.title("⚙️ Pengaturan")

file_upload = st.sidebar.file_uploader("Upload CSV sendiri (opsional)", type="csv")
if file_upload is not None:
    df = pd.read_csv(file_upload)
    st.sidebar.success("Memakai file upload")
else:
    df = muat_data_default(DATA_DEFAULT)
    st.sidebar.info("Memakai data_mahasiswa.csv")

kolom_numerik = df.select_dtypes(include="number").columns.tolist()
kolom_kategori = [
    k for k in df.select_dtypes(exclude="number").columns if df[k].nunique() <= 10
]

if not kolom_numerik:
    st.error("File ini tidak punya kolom numerik. Coba upload file lain.")
    st.stop()

kolom = st.sidebar.selectbox("Kolom yang dianalisis", kolom_numerik)

st.sidebar.subheader("Filter data")
df_f = df.copy()
for k in kolom_kategori:
    semua = sorted(df[k].dropna().unique())
    pilihan = st.sidebar.multiselect(f"{k}", semua, default=semua)
    df_f = df_f[df_f[k].isin(pilihan)]

x = df_f[kolom].dropna()
if x.empty:
    st.warning("Tidak ada data setelah difilter. Longgarkan filter di sidebar.")
    st.stop()

stat = hitung_statistik(x)


# ======================================================================
# 4. HALAMAN UTAMA
# ======================================================================
st.title("📊 Dashboard Statistik Deskriptif")
st.caption(f"Kolom: **{kolom}** | Data terpilih: {len(df_f)} dari {len(df)} baris")

tab_data, tab_stat, tab_vis, tab_dadu = st.tabs(
    ["📋 Data", "🧮 Statistik", "📈 Visualisasi", "🎲 Simulasi Dadu"]
)

# ----------------------------- TAB DATA -------------------------------
with tab_data:
    c1, c2, c3 = st.columns(3)
    c1.metric("Jumlah baris", len(df_f))
    c2.metric("Jumlah kolom", df_f.shape[1])
    c3.metric("Data kosong (total)", int(df_f.isna().sum().sum()))

    st.dataframe(df_f, width="stretch")

    with st.expander("Lihat data kosong per kolom"):
        st.dataframe(df_f.isna().sum().rename("Jumlah kosong"))

# --------------------------- TAB STATISTIK ----------------------------
with tab_stat:
    st.subheader("Ukuran Pemusatan")
    a, b, c, d = st.columns(4)
    a.metric("Mean", f"{stat['mean']:.2f}")
    b.metric("Median", f"{stat['median']:.2f}")
    c.metric("Modus", format_modus(stat["modus"]))
    d.metric("Jumlah data (n)", stat["n"])

    st.subheader("Ukuran Penyebaran")
    e, f, g, h = st.columns(4)
    e.metric("Std Dev (sampel)", f"{stat['std']:.2f}")
    f.metric("Range", f"{stat['max'] - stat['min']:.2f}")
    g.metric("IQR", f"{stat['iqr']:.2f}")
    h.metric("Jumlah outlier", len(stat["outlier"]))

    # Interpretasi otomatis: bandingkan mean dan median
    selisih = stat["mean"] - stat["median"]
    if abs(selisih) < 0.1 * stat["std"]:
        st.success("Mean ≈ Median → distribusi relatif **simetris**.")
    elif selisih > 0:
        st.info("Mean > Median → distribusi **condong ke kanan** (ada nilai besar yang menarik mean).")
    else:
        st.info("Mean < Median → distribusi **condong ke kiri** (ada nilai kecil yang menarik mean).")

    with st.expander("Detail kuartil dan batas outlier (aturan 1,5 × IQR)"):
        st.write(f"Q1 = {stat['q1']:.2f} | Q3 = {stat['q3']:.2f}")
        st.write(f"Batas bawah = {stat['batas_bawah']:.2f} | Batas atas = {stat['batas_atas']:.2f}")
        if len(stat["outlier"]) > 0:
            st.write("Nilai outlier:", stat["outlier"].tolist())
        else:
            st.write("Tidak ada outlier.")

    st.subheader("Ringkasan semua kolom numerik")
    st.dataframe(df_f[kolom_numerik].describe().round(2), width="stretch")

# --------------------------- TAB VISUALISASI --------------------------
with tab_vis:
    kiri, kanan = st.columns(2)

    with kiri:
        st.subheader("Histogram")
        n_bins = st.slider("Jumlah bin", 5, 30, 10)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(x, bins=n_bins, edgecolor="black")
        ax.axvline(stat["mean"], color="red", linestyle="--", label=f"Mean = {stat['mean']:.1f}")
        ax.axvline(stat["median"], color="green", label=f"Median = {stat['median']:.1f}")
        ax.set_xlabel(kolom)
        ax.set_ylabel("Frekuensi")
        ax.legend()
        st.pyplot(fig)

    with kanan:
        st.subheader("Boxplot")
        opsi_grup = ["(tanpa pengelompokan)"] + kolom_kategori
        grup = st.selectbox("Kelompokkan berdasarkan", opsi_grup)
        fig2, ax2 = plt.subplots(figsize=(6, 4))
        if grup == opsi_grup[0]:
            ax2.boxplot(x)
            ax2.set_xticks([1])
            ax2.set_xticklabels([kolom])
        else:
            label, data = [], []
            for nama_grup, isi in df_f.groupby(grup):
                nilai = isi[kolom].dropna()
                if len(nilai) > 0:
                    label.append(str(nama_grup))
                    data.append(nilai)
            ax2.boxplot(data)
            ax2.set_xticks(range(1, len(label) + 1))
            ax2.set_xticklabels(label)
            ax2.set_xlabel(grup)
        ax2.set_ylabel(kolom)
        st.pyplot(fig2)

    st.caption("Tip: bandingkan posisi garis mean dan median pada histogram dengan bentuk boxplot.")

# ---------------------------- TAB DADU --------------------------------
with tab_dadu:
    st.subheader("Simulasi Lempar Dadu: Hukum Bilangan Besar")
    st.write(
        "Peluang teori tiap sisi dadu adalah 1/6 ≈ 0,167. "
        "Coba ubah jumlah lemparan dan lihat apakah proporsi hasil makin mendekati peluang teori."
    )

    n_lempar = st.slider("Jumlah lemparan", 10, 100000, 100, step=10)

    if st.button("🎲 Lempar dadu!"):
        st.session_state["hasil_dadu"] = np.random.randint(1, 7, size=n_lempar)

    hasil = st.session_state.get("hasil_dadu")
    if hasil is None:
        st.info("Tekan tombol **Lempar dadu!** untuk memulai simulasi.")
    else:
        frek = np.bincount(hasil, minlength=7)[1:]       # frekuensi sisi 1..6
        proporsi = frek / len(hasil)
        penyimpangan = np.abs(proporsi - 1 / 6).max()

        m1, m2 = st.columns(2)
        m1.metric("Total lemparan", f"{len(hasil):,}")
        m2.metric("Penyimpangan terbesar dari 1/6", f"{penyimpangan:.4f}")

        fig3, ax3 = plt.subplots(figsize=(7, 3.5))
        ax3.bar(range(1, 7), proporsi)
        ax3.axhline(1 / 6, color="red", linestyle="--", label="Peluang teori (1/6)")
        ax3.set_xlabel("Sisi dadu")
        ax3.set_ylabel("Proporsi")
        ax3.set_ylim(0, max(0.35, proporsi.max() * 1.15))
        ax3.legend()
        st.pyplot(fig3)

        tabel = pd.DataFrame({
            "Sisi": range(1, 7),
            "Frekuensi": frek,
            "Proporsi": proporsi.round(4),
        })
        st.dataframe(tabel, hide_index=True)

st.divider()
st.caption("Statistik dan Probabilitas 1 - Pertemuan 4")

Jalankan:
```
streamlit run app.py
```

### Cek pemahaman
1. Pada `hitung_statistik()`, mengapa nilai `modus` disimpan sebagai **list**, bukan angka tunggal?
   *(Karena data bisa punya lebih dari satu modus, contoh `Jam_Belajar` memiliki modus 5 dan 9.)*
2. Apa yang terjadi bila semua pilihan pada filter `Kelas` dihapus? Baris kode mana yang menanganinya?
   *(`if x.empty: st.warning(...); st.stop()`. Tanpa ini aplikasi akan error.)*
3. Mengapa hasil lemparan dadu disimpan di `st.session_state`, bukan variabel biasa?
   *(Supaya hasil tetap tampil saat pengguna menggeser slider atau berinteraksi tanpa melempar ulang.)*
4. Mengapa batas outlier dihitung dengan `1,5 × IQR`, dan apa arti notasi `x[(x < bawah) | (x > atas)]`?
   *(Boolean masking dari Pertemuan 2; tanda `|` berarti "atau".)*
5. Coba **upload `data_penjualan_kafe.csv`** di sidebar. Apakah dashboard tetap bekerja? Apa yang berubah pada pilihan kolom dan filter?

### Tantangan tambahan (opsional)
- Tambahkan `st.download_button` untuk mengunduh data hasil filter.
- Tambahkan tab baru **"Korelasi"** berisi scatter plot dua kolom numerik pilihan pengguna dan nilai korelasinya (`df[[a, b]].corr()`).

## 7. Dashboard yang Baik: Evaluasi Dashboard Kita

Kita ulas kembali prinsip dari Pertemuan 3 terhadap dashboard yang baru dibuat:

| Prinsip | Penerapan di dashboard kita |
|---|---|
| Pahami audiens dan kebutuhannya | Filter di sidebar untuk dosen/mahasiswa yang ingin membandingkan kelompok |
| Batasi jumlah komponen visual | Hanya 2 grafik utama (histogram dan boxplot) + 1 grafik simulasi, dipisah per tab |
| *Load time* singkat | `@st.cache_data` pada pembacaan data |
| Informasi kunci di posisi utama | Angka statistik (`st.metric`) ada di bagian atas tab Statistik |
| Hindari visualisasi menyesatkan | Histogram dan boxplot jujur terhadap skala data; grafik dadu memakai garis acuan 1/6 |

**Latihan berpikir:** pada grafik batang, bila sumbu-y **tidak dimulai dari 0**, selisih kecil tampak besar. Di dashboard kita, grafik mana yang paling rawan menyesatkan jika sumbunya dipotong? Mengapa?

## 8. Troubleshooting Umum

| Gejala | Penyebab / solusi |
|---|---|
| `streamlit: command not found` | Jalankan `python -m streamlit run app.py` |
| `ModuleNotFoundError: No module named '...'` | Install dulu: `pip install nama_library` |
| `FileNotFoundError: data/...csv` | Folder `data/` harus **sejajar** dengan file `.py`. Pastikan struktur folder benar |
| Port 8501 sedang dipakai | `streamlit run app.py --server.port 8502` |
| Perubahan kode tidak muncul | Klik **Rerun** / pilih **Always rerun** di pojok kanan atas browser |
| Muncul peringatan `use_container_width` | Gunakan `width="stretch"` dan update dengan `pip install -U streamlit` |
| Aplikasi "macet" | Tekan `Ctrl + C` di terminal, lalu jalankan ulang |

## 9. Rangkuman (Cheat Sheet)

| Kebutuhan | Kode |
|---|---|
| Menjalankan aplikasi | `streamlit run app.py` |
| Judul dan teks | `st.title()`, `st.header()`, `st.write()`, `st.markdown()` |
| Menampilkan tabel | `st.dataframe(df)` |
| Menampilkan angka kunci | `st.metric("Mean", f"{x.mean():.2f}")` |
| Menampilkan grafik Matplotlib | `st.pyplot(fig)` |
| Input pengguna | `st.slider`, `st.selectbox`, `st.multiselect`, `st.button`, `st.file_uploader` |
| Tata letak | `st.sidebar`, `st.columns`, `st.tabs`, `st.expander` |
| Menyimpan nilai antar-rerun | `st.session_state["kunci"]` |
| Menghemat waktu muat data | `@st.cache_data` |
| Menghentikan eksekusi aman | `st.stop()` |

**Pesan utama:**
1. Streamlit menjalankan ulang seluruh skrip setiap ada interaksi.
2. Widget mengembalikan **nilai Python biasa**, sehingga semua yang sudah kita pelajari (if, loop, function, NumPy, Pandas) bisa dipakai langsung.
3. Selalu tangani kasus ekstrem: data kosong, filter kosong, input salah.
4. Dashboard yang baik **sederhana, cepat, dan jujur**.

---
## 10. Kuis Latihan

> **Cara pengerjaan:** jalankan cell `%%writefile` untuk membuat file kerangka, lengkapi bagian `TODO` di file tersebut (boleh di VS Code), lalu jalankan dengan `streamlit run nama_file.py`.
> **Pengumpulan:** file `.py` + 1 screenshot tampilan aplikasi untuk tiap kuis.

### Kuis 1: Kalkulator Statistik Mini
Buat aplikasi tempat pengguna mengetik angka dipisah koma (contoh `70, 85, 90, 65, 78`), lalu menekan tombol **Hitung** untuk melihat **Mean, Median, Std Dev (sampel), Min, dan Max**.

Syarat:
1. Perhitungan hanya berjalan saat tombol ditekan.
2. Jika ada input yang bukan angka (contoh `70, abc, 90`), tampilkan pesan **error** dan hentikan aplikasi dengan aman.
3. Jika data kurang dari 2 angka, tampilkan **warning**.
4. Tampilkan hasil dengan `st.metric` dalam 5 kolom.

Uji dengan `70, 85, 90, 65, 78`. Hasil yang benar: Mean 77,60, Median 78, Std Dev 10,31, Min 65, Max 90.

In [ ]:
%%writefile kuis1.py
import numpy as np
import streamlit as st

st.title("🧮 Kalkulator Statistik Mini")
st.write("Masukkan angka-angka yang dipisah koma. Contoh: 70, 85, 90, 65, 78")

teks = st.text_input("Data angka")

# TODO 1: buat tombol "Hitung".
#         Semua langkah di bawah dikerjakan HANYA jika tombol ditekan.
# TODO 2: ubah teks menjadi list angka (float) dengan split(",").
#         Gunakan try-except: jika ada yang bukan angka, tampilkan st.error(...) lalu st.stop().
# TODO 3: jika angka kurang dari 2 buah, tampilkan st.warning(...) lalu st.stop().
# TODO 4: ubah ke np.array, lalu tampilkan Mean, Median, Std Dev (sampel, ddof=1),
#         Min, dan Max memakai st.metric di dalam 5 kolom (st.columns).

### Kuis 2: Dashboard Penjualan Kafe
Dataset `data/data_penjualan_kafe.csv` berisi penjualan 3 menu selama 30 hari (kolom: `Tanggal`, `Menu`, `Harga`, `Jumlah_Terjual`, `Pendapatan`). Lengkapi kerangka dashboard berikut sesuai instruksi `TODO` 1 sampai 7:

1. Fungsi `hitung_outlier()` (aturan 1,5 × IQR).
2. Filter menu di sidebar.
3. Empat `st.metric`: Total Pendapatan, Rata-rata Pendapatan Harian, Hari Pendapatan Tertinggi, dan Jumlah Outlier (dideteksi **per menu**).
4. Grafik garis pendapatan harian.
5. Boxplot pendapatan per menu.
6. Kesimpulan singkat dengan `st.info`.
7. **Pertanyaan konseptual:** apa yang terjadi jika outlier dideteksi pada seluruh data yang digabung (tidak per menu)? Jelaskan alasannya.

*Petunjuk:* ada satu hari saat terjadi lonjakan penjualan luar biasa. Dashboard-mu harus bisa menemukannya.

In [ ]:
%%writefile kuis2.py
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Dashboard Kafe", page_icon="☕", layout="wide")

df = pd.read_csv(Path(__file__).parent / "data" / "data_penjualan_kafe.csv")

st.title("☕ Dashboard Penjualan Kafe")


def hitung_outlier(x):
    """Kembalikan nilai outlier (aturan 1,5 x IQR) dari sebuah Series."""
    # TODO 1: lengkapi fungsi ini
    pass


# TODO 2: di sidebar, buat multiselect "Menu" (default: semua menu),
#         lalu filter df sehingga hanya berisi menu yang dipilih (simpan di df_f).

# TODO 3: tampilkan 4 st.metric dalam 4 kolom:
#         (a) Total Pendapatan  (format: Rp1.234.567)
#         (b) Rata-rata Pendapatan Harian (total semua menu per tanggal, lalu di-mean)
#         (c) Tanggal dengan pendapatan harian tertinggi
#         (d) Jumlah outlier pada kolom Pendapatan (dideteksi PER MENU, lalu dijumlahkan)

# TODO 4: tampilkan grafik garis (st.line_chart) pendapatan harian total (index = Tanggal).

# TODO 5: buat boxplot Pendapatan per Menu (matplotlib + st.pyplot).

# TODO 6: tampilkan st.info(...) berisi kesimpulan singkat: menu mana yang punya outlier?

# TODO 7 (konseptual, tulis jawabanmu sebagai komentar di sini):
#         Coba hitung outlier pada kolom Pendapatan dengan SELURUH menu digabung (tanpa dipisah per menu).
#         Apakah lonjakan penjualan Es Teh terdeteksi? Mengapa hasilnya berbeda dengan deteksi per menu?

---
### Preview Pertemuan 5
Sampai Pertemuan 4 kita fokus pada **statistika deskriptif**: meringkas dan menampilkan data. Mulai Pertemuan 5 kita masuk ke sisi **probabilitas**: ruang sampel, kejadian, aturan peluang, dan simulasinya di Python. Simulasi dadu di tab terakhir dashboard tadi adalah pembukanya.

*Selesai! Jangan lupa simpan semua file dan kumpulkan sesuai instruksi pengajar.*